# 🤖 Agente principal + Agentes especializados — Publicaciones de LinkedIn

**Reto proyecto — Desarrollo de Soluciones IA**

Chatbot de terminal con el **OpenAI Agents SDK**: un **agente principal** recibe las solicitudes y las **delega (handoff)** al agente especializado según la temática (**marketing**, **programación** o **jurídico-legal**). Cada especialista genera una publicación de LinkedIn con **salida estructurada Pydantic**: `title`, `content`, `hashtags`, `category`. Usa `gpt-5` vía tu recurso de Azure.

### Mapeo de la estructura pedida → celdas

```
├── main.py                        → Celda «Opción B (run_cli)»
├── agents/
│   ├── main_agent.py              → Celda «Agente principal»
│   └── specialized_agents.py      → Celda «Agentes especializados»
├── core/
│   ├── chatbot.py                 → Celda «Chatbot»
│   └── conversation.py            → Celda «Gestión del historial»
├── requirements.txt               → Celda de dependencias
├── .env                           → Credenciales (privadas)
└── README.md                      → Este encabezado
```

### Cómo funciona la delegación

1. El usuario pide, p. ej., *"Haz un post sobre buenas prácticas en Python"*.
2. El **agente principal** analiza la temática y hace un **handoff** al especialista adecuado (aquí, Programación). Para el SDK, cada handoff es una herramienta `transfer_to_<agente>`.
3. El **especialista** genera la publicación con `output_type=LinkedinPost` (Pydantic), garantizando la estructura título/contenido/hashtags/categoría.
4. La CLI muestra **qué agente** ha procesado cada consulta y mantiene el **historial** de la sesión.

### Salidas estructuradas (`output_type`)

Cada agente especializado se crea con `Agent(..., output_type=LinkedinPost)`. Esto activa
los **structured outputs** del SDK: el modelo está obligado a devolver una respuesta que
encaja con el esquema del modelo Pydantic `LinkedinPost`, y el SDK la valida y la entrega
ya como objeto Python (no como texto a parsear a mano). Es el mismo patrón que usarías con
`response_format`/`text_format` en la API de OpenAI directamente, pero integrado en el
Agent. Si el modelo no lleva `output_type`, `result.final_output` sería simplemente texto
libre; por eso los tres especialistas SÍ lo llevan, y el agente principal NO (él solo
delega o pide aclaraciones en texto libre).

### Configuración (`.env`)

```dotenv
AZURE_OPENAI_ENDPOINT=https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/
AZURE_OPENAI_API_KEY=tu_clave_de_azure
OPENAI_MODEL=gpt-5
```

### Cómo ejecutar
Celdas de arriba abajo: dependencias → configuración → modelo Pydantic → agentes → historial → chatbot → **Opción A** (demos directas) u **Opción B** (CLI interactiva con `/salir`, `/ayuda`, `/reiniciar`).


## 1. Dependencias (`requirements.txt`)

```text
openai-agents
python-dotenv
```

> El paquete del OpenAI Agents SDK se llama **`openai-agents`** (se importa como `agents`). Requiere Python 3.10+.


In [7]:
# Instalación de dependencias (ejecutar una vez y reiniciar el kernel si es la primera vez)
%pip install -q openai-agents python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuración (`.env` + cliente del SDK)

El Agents SDK usa por defecto la API de OpenAI de platform.openai.com. Para usar **tu recurso de Azure** (endpoint compatible `/openai/v1/`), se configura un `AsyncOpenAI` global con `set_default_openai_client`. También se **desactiva el tracing** (requiere una clave de platform.openai.com que no usamos).


In [8]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import set_default_openai_client, set_tracing_disabled

load_dotenv()

# Endpoint compatible OpenAI del recurso de Azure (acaba en /openai/v1/)
_raw = os.getenv("AZURE_OPENAI_ENDPOINT",
                 "https://marcvancutseme7172-2656-resource.services.ai.azure.com/")
_base = _raw.rstrip("/")
for _suf in ("/openai/v1", "/openai"):
    if _base.endswith(_suf):
        _base = _base[: -len(_suf)]
OPENAI_BASE_URL = _base + "/openai/v1/"
OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5")

if OPENAI_API_KEY:
    cliente_azure = AsyncOpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)
    set_default_openai_client(cliente_azure)   # todos los agentes usarán este cliente
    set_tracing_disabled(True)                 # el tracing necesita clave de platform.openai.com
    print("✅ Agents SDK apuntando a Azure:")
    print(f"   Endpoint: {OPENAI_BASE_URL}")
    print(f"   Modelo:   {OPENAI_MODEL}")
else:
    print("❌ Falta AZURE_OPENAI_API_KEY. Crea un .env con:")
    print("   AZURE_OPENAI_API_KEY=tu_clave")
    print(f"   AZURE_OPENAI_ENDPOINT={_raw}")


✅ Agents SDK apuntando a Azure:
   Endpoint: https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/
   Modelo:   gpt-5


## 3. Salida estructurada con Pydantic (`LinkedinPost`)

Todas las publicaciones siguen la misma estructura: **título**, **contenido**, **hashtags** y **categoría**. Con `output_type=LinkedinPost`, el SDK usa *structured outputs*: la respuesta del especialista llega **ya validada** como objeto Pydantic.


In [9]:
from enum import Enum
from pydantic import BaseModel, ConfigDict, Field, field_validator


class CategoriaPost(str, Enum):
    """Temáticas disponibles (una por agente especializado)."""
    MARKETING = "marketing"
    PROGRAMACION = "programacion"
    JURIDICO_LEGAL = "juridico-legal"


class LinkedinPost(BaseModel):
    """Publicación de LinkedIn con estructura fija."""
    model_config = ConfigDict(extra="forbid")   # additionalProperties: false (modo estricto)

    title: str = Field(description="Encabezado atractivo de la publicación")
    content: str = Field(description="Cuerpo principal de la publicación, con saltos de línea")
    hashtags: list[str] = Field(description="Palabras clave relevantes, con o sin '#'")
    category: CategoriaPost = Field(description="Temática de la publicación")

    # Nota: en structured outputs no se aplican minLength/maxItems en el esquema,
    # así que validamos longitudes/formato con field_validators (post-validación).
    @field_validator("title")
    @classmethod
    def _titulo_valido(cls, v: str) -> str:
        v = v.strip()
        if not v:
            raise ValueError("El título no puede estar vacío.")
        if len(v) > 120:
            raise ValueError("El título no puede superar los 120 caracteres.")
        return v

    @field_validator("content")
    @classmethod
    def _contenido_minimo(cls, v: str) -> str:
        v = v.strip()
        if len(v) < 50:
            raise ValueError("El contenido debe tener al menos 50 caracteres.")
        return v

    @field_validator("hashtags")
    @classmethod
    def _normalizar_hashtags(cls, v: list[str]) -> list[str]:
        limpios = []
        for h in v:
            h = h.strip().lstrip("#").replace(" ", "")
            if h:
                limpios.append(f"#{h}")
        if not limpios:
            raise ValueError("Debe incluir al menos un hashtag.")
        if len(limpios) > 8:
            raise ValueError("No se permiten más de 8 hashtags.")
        return limpios


def post_a_dict(post: LinkedinPost) -> dict:
    """Serializa la publicación a un dict plano (útil para persistir o enviar a otros sistemas)."""
    return post.model_dump(mode="json")


def post_a_json(post: LinkedinPost) -> str:
    """Serializa la publicación a una cadena JSON."""
    return post.model_dump_json(indent=2)


def mostrar_post(post: LinkedinPost) -> str:
    """Formatea una publicación para mostrarla en el terminal."""
    lineas = [
        "─" * 56,
        f"📌 {post.title}",
        "─" * 56,
        post.content,
        "",
        "🏷️  " + " ".join(post.hashtags),
        f"📂 Categoría: {post.category.value}",
        "─" * 56,
    ]
    return "\n".join(lineas)


## 4. Agentes especializados (`agents/specialized_agents.py`)

Tres especialistas —**Marketing**, **Programación** y **Jurídico-Legal**— con instrucciones propias, few-shot de estilo y `output_type=LinkedinPost`. El `handoff_description` le dice al agente principal cuándo delegar en cada uno.


In [10]:
from agents import Agent

_INSTRUCCIONES_COMUNES = (
    "Eres un experto creando publicaciones de LinkedIn en ESPAÑOL. "
    "Genera UNA publicación completa siguiendo la estructura pedida (título, contenido, "
    "hashtags, categoría). El contenido debe ser profesional, con gancho inicial, "
    "2-4 párrafos cortos o viñetas, y una llamada a la acción final. "
    "Usa un tono cercano pero riguroso, y emojis con moderación (2-5). "
    "Los hashtags: entre 4 y 8, relevantes y específicos del tema. "
    "Requisitos de longitud y estructura del contenido (más allá del mínimo técnico): "
    "al menos 3 párrafos o bloques (gancho + desarrollo + cierre/CTA), y un mínimo "
    "aproximado de 400 caracteres en total, para que la publicación tenga sustancia y no "
    "se quede en una frase suelta. Nunca generes menos de 50 caracteres: sería rechazado."
)

agente_marketing = Agent(
    name="Agente de Marketing",
    handoff_description=(
        "Especialista en publicaciones de LinkedIn sobre marketing: estrategia, redes "
        "sociales, branding, publicidad, SEO/SEM, email marketing, growth, ventas."
    ),
    instructions=(
        f"{_INSTRUCCIONES_COMUNES} "
        "Tu temática es MARKETING: estrategia digital, marca, campañas, métricas (CTR, CAC, "
        "LTV), tendencias. La categoría de la publicación debe ser 'marketing'.\n\n"
        "Ejemplo de estilo esperado (no lo copies literalmente, es solo referencia de tono "
        "y estructura):\n"
        "Título: 'Por qué tu CAC sube y nadie te lo dice'\n"
        "Contenido: '¿Sigues midiendo el éxito de una campaña solo por el alcance? 📉\\n\\n"
        "El coste de adquisición de cliente (CAC) es la métrica que de verdad decide si una "
        "estrategia es rentable. Muchas marcas optimizan CTR y olvidan el funnel completo.\\n\\n"
        "3 señales de que tu CAC se está disparando:\\n"
        "• El CTR sube pero las conversiones no.\\n"
        "• El coste por lead crece cada trimestre.\\n"
        "• La retención no compensa la inversión en captación.\\n\\n"
        "¿Cómo mides tú el retorno real de tus campañas? 👇'\n"
        "Hashtags: ['#marketingdigital', '#estrategia', '#CAC', '#growth']\n\n"
        "Ejemplos de peticiones típicas que deberías saber cubrir con este estilo:\n"
        "- 'Post sobre el lanzamiento de una campaña en redes sociales'\n"
        "- 'Publicación sobre por qué el branding importa más que el precio'\n"
        "- 'Algo sobre tendencias de marketing de contenidos para 2026'"
    ),
    output_type=LinkedinPost,
    model=OPENAI_MODEL,
)

agente_programacion = Agent(
    name="Agente de Programación",
    handoff_description=(
        "Especialista en publicaciones de LinkedIn sobre programación y tecnología: "
        "lenguajes, frameworks, buenas prácticas, arquitectura, IA, DevOps, carrera dev."
    ),
    instructions=(
        f"{_INSTRUCCIONES_COMUNES} "
        "Tu temática es PROGRAMACIÓN: desarrollo de software, lenguajes, buenas prácticas, "
        "herramientas, IA aplicada, productividad del desarrollador. La categoría debe ser "
        "'programacion'.\n\n"
        "Ejemplo de estilo esperado (no lo copies literalmente, es solo referencia de tono "
        "y estructura):\n"
        "Título: 'El código limpio no es opcional, es deuda técnica disfrazada'\n"
        "Contenido: 'Llevo años viendo el mismo error: se prioriza \"que funcione\" sobre "
        "\"que se entienda\". 🧑\\u200d💻\\n\\n"
        "El código que escribes hoy lo va a leer otra persona (o tú mismo) dentro de 6 meses. "
        "Si tarda 10 minutos en entender una función, algo falla.\\n\\n"
        "3 hábitos que marcan la diferencia:\\n"
        "• Nombres que explican el 'qué', no el 'cómo'.\\n"
        "• Funciones pequeñas con una sola responsabilidad.\\n"
        "• Tests que documentan el comportamiento esperado.\\n\\n"
        "¿Qué hábito de código limpio te costó más adoptar? 👇'\n"
        "Hashtags: ['#programacion', '#cleancode', '#buenaspracticas', '#desarrollosoftware']\n\n"
        "Ejemplos de peticiones típicas que deberías saber cubrir con este estilo:\n"
        "- 'Post sobre las ventajas de usar TypeScript frente a JavaScript'\n"
        "- 'Publicación sobre cómo la IA está cambiando el día a día de un developer'\n"
        "- 'Algo sobre por qué hacer code review mejora la calidad del equipo'"
    ),
    output_type=LinkedinPost,
    model=OPENAI_MODEL,
)

agente_juridico = Agent(
    name="Agente Jurídico-Legal",
    handoff_description=(
        "Especialista en publicaciones de LinkedIn sobre temas jurídicos y legales: "
        "normativa, RGPD, contratos, laboral, mercantil, compliance, propiedad intelectual."
    ),
    instructions=(
        f"{_INSTRUCCIONES_COMUNES} "
        "Tu temática es JURÍDICO-LEGAL: novedades normativas, protección de datos, derecho "
        "laboral/mercantil, compliance. Divulga sin dar asesoramiento legal vinculante "
        "(añade un matiz de 'esto no es asesoramiento legal' cuando proceda). La categoría "
        "debe ser 'juridico-legal'.\n\n"
        "Ejemplo de estilo esperado (no lo copies literalmente, es solo referencia de tono "
        "y estructura):\n"
        "Título: '¿Tu empresa trata datos de clientes? Esto exige el RGPD'\n"
        "Contenido: 'Muchas pymes creen que el RGPD solo afecta a grandes corporaciones. "
        "No es así. ⚖️\\n\\n"
        "Si recoges nombres, emails o cualquier dato personal, tienes obligaciones legales, "
        "aunque seas un equipo pequeño.\\n\\n"
        "3 puntos que no puedes pasar por alto:\\n"
        "• Base legal clara para cada tratamiento de datos.\\n"
        "• Registro de actividades de tratamiento actualizado.\\n"
        "• Procedimiento definido ante una posible brecha de seguridad.\\n\\n"
        "Este contenido es divulgativo y no sustituye el asesoramiento legal de un "
        "profesional. ¿Tu empresa ya tiene esto revisado? 👇'\n"
        "Hashtags: ['#RGPD', '#proteccionDatos', '#compliance', '#derechodigital']\n\n"
        "Ejemplos de peticiones típicas que deberías saber cubrir con este estilo:\n"
        "- 'Post sobre los derechos laborales del teletrabajo'\n"
        "- 'Publicación sobre cláusulas abusivas en contratos mercantiles'\n"
        "- 'Algo sobre la nueva normativa de compliance para empresas'"
    ),
    output_type=LinkedinPost,
    model=OPENAI_MODEL,
)

AGENTES_ESPECIALIZADOS = [agente_marketing, agente_programacion, agente_juridico]
print("✅ Especialistas creados:", ", ".join(a.name for a in AGENTES_ESPECIALIZADOS))


✅ Especialistas creados: Agente de Marketing, Agente de Programación, Agente Jurídico-Legal


## 5. Agente principal (`agents/main_agent.py`)

Recibe la solicitud del usuario y **delega** al especialista adecuado mediante **handoffs** (para el modelo, herramientas `transfer_to_<agente>`). Si la temática no encaja o falta información, responde él mismo pidiendo aclaración.


In [11]:
from agents import handoff

# El nombre de la herramienta de handoff se deriva de Agent.name ("transfer_to_<nombre>").
# Como nuestros nombres tienen espacios/tildes (para mostrarlos bonitos en la CLI), el SDK
# los sanea y avisa. Se lo damos explícito, ya limpio, con handoff(tool_name_override=...).
_handoffs = [
    handoff(agente_marketing, tool_name_override="transfer_to_marketing"),
    handoff(agente_programacion, tool_name_override="transfer_to_programacion"),
    handoff(agente_juridico, tool_name_override="transfer_to_juridico_legal"),
]

agente_principal = Agent(
    name="Agente Principal",
    instructions=(
        "Eres el coordinador de un equipo que crea publicaciones de LinkedIn en español. "
        "Analiza la solicitud del usuario y DELEGA en el especialista adecuado según la "
        "temática:\n"
        "- Marketing (estrategia, redes, marca, ventas) → Agente de Marketing.\n"
        "- Programación/tecnología (código, lenguajes, IA, DevOps) → Agente de Programación.\n"
        "- Jurídico-legal (normativa, RGPD, contratos, laboral) → Agente Jurídico-Legal.\n\n"
        "Casos límite (temática ambigua o mixta): elige el especialista según el ENFOQUE "
        "PRINCIPAL de la solicitud, no según todas las palabras que mencione.\n"
        "- 'Post sobre cómo el RGPD afecta a las campañas de email marketing' → el eje es "
        "cumplimiento normativo → Jurídico-Legal.\n"
        "- 'Post sobre cómo vender mejor un producto de software con buen storytelling' → "
        "el eje es venta/estrategia → Marketing (aunque mencione 'software').\n"
        "- 'Post sobre por qué los developers deberían entender los contratos de licencia "
        "de código abierto' → el eje es técnico/carrera dev → Programación.\n"
        "- 'Post sobre la nueva ley de IA y su impacto en las empresas tech' → el eje es "
        "normativo → Jurídico-Legal (aunque hable de tecnología).\n"
        "- 'Post sobre cómo usar LinkedIn Ads para vender un curso de programación' → el eje "
        "es venta/promoción → Marketing (aunque el producto sea de programación).\n"
        "- 'Post sobre qué contenido técnico atrae más talento developer a una empresa' → si "
        "el foco es reclutamiento/employer branding → Marketing; si el foco es qué hace "
        "buen contenido técnico en sí (calidad, buenas prácticas) → Programación. Fíjate en "
        "el verbo principal de la petición: 'atraer/vender/promocionar' → Marketing; "
        "'explicar/enseñar/mejorar' un tema técnico → Programación.\n"
        "Si tras aplicar este criterio sigues dudando entre dos, delega en el que tenga más "
        "peso en la frase (el sustantivo principal del tema, no los adjetivos o el sector). "
        "Solo si NO encaja en ninguna de las tres o falta información esencial para elegir, "
        "responde tú brevemente en español indicando las temáticas disponibles o pidiendo la "
        "aclaración necesaria. No generes tú las publicaciones: eso es trabajo de los "
        "especialistas."
    ),
    handoffs=_handoffs,
    model=OPENAI_MODEL,
)
print(f"✅ '{agente_principal.name}' creado con {len(agente_principal.handoffs)} handoffs.")


✅ 'Agente Principal' creado con 3 handoffs.


### 🧪 Demo automatizada: validar la clasificación de temáticas

Ejecuta una consulta representativa de cada temática y comprueba que el **handoff** termina en el especialista esperado (`result.last_agent`). Es una prueba repetible de la delegación del agente principal, sin depender del juicio manual. Requiere conexión real (llama al modelo).

In [12]:
from agents import Runner  # se usa aquí; también se reutiliza en el Chatbot más abajo

casos_prueba = [
    ("Post sobre cómo mejorar el CTR de una campaña de email marketing", "Agente de Marketing"),
    ("Publicación sobre buenas prácticas de testing en Python", "Agente de Programación"),
    ("Post sobre las obligaciones de una empresa según el RGPD", "Agente Jurídico-Legal"),
]

aciertos = 0
for consulta, esperado in casos_prueba:
    result = await Runner.run(agente_principal, [{"role": "user", "content": consulta}])
    obtenido = result.last_agent.name if result.last_agent else "(agente principal)"
    ok = obtenido == esperado
    aciertos += ok
    icono = "✅" if ok else "❌"
    print(f"{icono} '{consulta[:55]}...' -> esperado: {esperado} | obtenido: {obtenido}")

print(f"\n{aciertos}/{len(casos_prueba)} clasificaciones correctas.")


✅ 'Post sobre cómo mejorar el CTR de una campaña de email ...' -> esperado: Agente de Marketing | obtenido: Agente de Marketing
✅ 'Publicación sobre buenas prácticas de testing en Python...' -> esperado: Agente de Programación | obtenido: Agente de Programación
✅ 'Post sobre las obligaciones de una empresa según el RGP...' -> esperado: Agente Jurídico-Legal | obtenido: Agente Jurídico-Legal

3/3 clasificaciones correctas.


## 6. Gestión del historial (`core/conversation.py`)

El SDK devuelve en cada ejecución la lista completa de ítems (`result.to_input_list()`). La clase `Conversation` guarda ese historial durante toda la sesión, de modo que cada nueva consulta **conserva el contexto** (p. ej. "hazlo más corto" se refiere al post anterior).


In [13]:
class Conversation:
    """Mantiene el historial de la conversación entre el usuario y los agentes."""

    def __init__(self, max_items: int = 60):
        self.items: list = []          # historial en formato del Agents SDK
        self.max_items = max_items     # límite para no crecer sin control
        self.turnos_usuario = 0

    def entrada_con(self, mensaje_usuario: str) -> list:
        """Historial + nuevo mensaje del usuario, listo para Runner.run()."""
        return self.items + [{"role": "user", "content": mensaje_usuario}]

    def actualizar(self, result) -> None:
        """Sustituye el historial por el de la última ejecución (incluye handoffs y salida)."""
        items = result.to_input_list()
        # Si crece demasiado, conservamos los últimos max_items elementos
        self.items = items[-self.max_items:] if len(items) > self.max_items else items
        self.turnos_usuario += 1

    def reiniciar(self) -> None:
        self.items.clear()
        self.turnos_usuario = 0

    def resumen(self) -> str:
        return f"{self.turnos_usuario} turnos, {len(self.items)} ítems en el historial"


## 7. Chatbot (`core/chatbot.py`)

Orquesta cada turno: pasa el historial + la consulta al **agente principal** con `Runner.run()`, detecta **qué agente** ha terminado procesando la consulta (`result.last_agent`) y formatea la salida (publicación estructurada o texto del principal).


In [14]:
from agents import Runner


class Chatbot:
    """Lógica principal: ejecuta turnos y gestiona errores y formato de salida."""

    def __init__(self, agente_inicial: "Agent", conversacion: "Conversation | None" = None,
                 max_turns: int = 6):
        self.agente_inicial = agente_inicial
        self.conversacion = conversacion or Conversation()
        self.max_turns = max_turns   # límite de pasos internos (handoffs, etc.) por consulta

    async def preguntar(self, consulta: str):
        """Ejecuta un turno completo.

        Returns:
            (texto_formateado, nombre_agente, post) — post es LinkedinPost o None.
            Si hay error, texto_formateado explica el problema y nombre_agente es None.
        """
        entrada = self.conversacion.entrada_con(consulta)
        try:
            result = await Runner.run(self.agente_inicial, entrada, max_turns=self.max_turns)
        except Exception as e:
            nombre = type(e).__name__
            if "MaxTurns" in nombre:
                return ("⚠️  La consulta necesitó demasiados pasos internos. "
                        "Prueba a formularla de forma más directa."), None, None
            if "AuthenticationError" in nombre:
                return "❌ Credenciales inválidas (401). Revisa AZURE_OPENAI_API_KEY.", None, None
            if "NotFoundError" in nombre:
                return (f"❌ Deployment no encontrado (404). ¿Existe '{OPENAI_MODEL}' "
                        "en tu recurso?"), None, None
            if "Connection" in nombre:
                return "❌ No se pudo conectar con el endpoint. Revisa tu conexión.", None, None
            return f"⚠️  Error al procesar la consulta ({nombre}: {e}).", None, None

        self.conversacion.actualizar(result)
        agente = result.last_agent.name if result.last_agent else "desconocido"
        salida = result.final_output

        if isinstance(salida, LinkedinPost):
            return mostrar_post(salida), agente, salida
        return str(salida), agente, None

    def reiniciar(self) -> None:
        self.conversacion.reiniciar()


## 8. Interfaz de terminal (`main.py`)

Bucle principal continuo con: **historial durante toda la sesión**, comandos **`/salir`** (salida limpia), **`/ayuda`** y **`/reiniciar`**, e **indicadores visuales** de qué agente procesa cada consulta.

> **Portabilidad a proyecto de archivos:** para pasar esto a `main.py` real, bastaría con mover `_ayuda_agentes` y `run_cli` a ese archivo y añadir al final:
> ```python
> if __name__ == "__main__":
>     import asyncio
>     asyncio.run(run_cli())
> ```


In [15]:
def _ayuda_agentes() -> None:
    print("\nℹ️  Genero publicaciones de LinkedIn. Temáticas y sus especialistas:")
    print("   • Marketing        → 'Haz un post sobre tendencias de email marketing'")
    print("   • Programación     → 'Publicación sobre buenas prácticas en Python'")
    print("   • Jurídico-Legal   → 'Post sobre las novedades del RGPD'")
    print("Mantengo el contexto: puedes pedir '/reiniciar' para empezar de cero.")
    print("Comandos:  /ayuda  ·  /reiniciar  ·  /salir\n")


async def run_cli(chatbot: "Chatbot | None" = None) -> None:
    """Aplicación de terminal interactiva (equivale a main.py)."""
    print("=" * 62)
    print("💼 Generador de publicaciones de LinkedIn — Agents SDK")
    print(f"   Agente principal + {len(AGENTES_ESPECIALIZADOS)} especialistas · modelo {OPENAI_MODEL}")
    print("=" * 62)

    if not OPENAI_API_KEY:
        print("❌ Falta AZURE_OPENAI_API_KEY en el .env.")
        return

    chatbot = chatbot or Chatbot(agente_principal)
    _ayuda_agentes()

    while True:
        try:
            consulta = input("🧑 Tú > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 ¡Hasta luego!")
            break

        if not consulta:
            continue
        comando = consulta.lower()
        if comando in ("/salir", "/exit", "/quit", "salir"):
            print(f"👋 ¡Hasta luego! ({chatbot.conversacion.resumen()})")
            break
        if comando in ("/ayuda", "/help"):
            _ayuda_agentes()
            continue
        if comando in ("/reiniciar", "/reset"):
            chatbot.reiniciar()
            print("🔄 Historial borrado. Empezamos de cero.")
            continue

        # Indicador visual: la consulta entra por el agente principal...
        print("⏳ Agente Principal analizando la solicitud y delegando...")
        respuesta, agente, post = await chatbot.preguntar(consulta)
        # ...y este indicador muestra qué agente la ha procesado finalmente
        if agente:
            icono = "🤝" if agente != agente_principal.name else "🧭"
            print(f"{icono} Procesado por: {agente}\n")
        print(respuesta)
        print(f"   💬 Sesión: {chatbot.conversacion.resumen()}")
        print()


### ▶️ Opción A — Consultas directas (sin `input()`, recomendado en notebooks)

Demuestra la delegación por temática y el mantenimiento del historial (la segunda consulta se apoya en el contexto de la primera).


In [16]:
bot = Chatbot(agente_principal)

for consulta in [
    "Quiero un post sobre buenas prácticas de código limpio en Python",
    "Ahora hazlo más corto y con un tono más informal",     # usa el historial
    "Un post sobre la importancia del RGPD para pymes",     # cambia de especialista
]:
    print(f"🧑 {consulta}")
    print("⏳ Agente Principal delegando...")
    respuesta, agente, post = await bot.preguntar(consulta)
    if agente:
        print(f"🤝 Procesado por: {agente}")
    print(respuesta)
    print()


🧑 Quiero un post sobre buenas prácticas de código limpio en Python
⏳ Agente Principal delegando...
🤝 Procesado por: Agente de Programación
────────────────────────────────────────────────────────
📌 Python limpio: menos magia, más intención
────────────────────────────────────────────────────────
Es tentador escribir una solución rápida y “pitónica”. Pero el código que importa es el que otros entienden sin preguntar. 🐍✨

Claves que aplico a diario:
• Nombrado explícito y consistente: evita abreviaturas crípticas; prefiero total_cost a tc.
• Funciones pequeñas y puras: una responsabilidad, un nivel de abstracción; delega con helpers.
• Tipado gradual con type hints y mypy/pyright: documentación viva que previene errores.
• Estructura de proyecto clara: src/, tests/, pyproject.toml; evita scripts sueltos.
• Manejo de errores con excepciones específicas y logging con niveles (INFO, WARNING, ERROR).
• Estilo automático: black + isort + ruff; deja que las herramientas discutan por ti.
• Test

### 📄 `main.py` (contenido de referencia para el proyecto de archivos)

La siguiente celda **no se ejecuta** dentro del notebook (aquí ya usamos `await run_cli()` directamente); es el contenido exacto que tendría `main.py` si separas el proyecto en archivos, tal como indica el enunciado.

In [17]:
# --- Contenido de referencia para main.py (proyecto de archivos separados) ---
#
# import asyncio
# from dotenv import load_dotenv
# from agents.main_agent import agente_principal      # ver: agents/main_agent.py
# from core.chatbot import Chatbot                     # ver: core/chatbot.py
# from core.cli_interface import run_cli                # ver: core/cli_interface.py (o cli_interface.py)
#
# if __name__ == "__main__":
#     load_dotenv()
#     asyncio.run(run_cli())


### ▶️ Opción B — CLI interactiva (`main.py`)


In [18]:
await run_cli()


💼 Generador de publicaciones de LinkedIn — Agents SDK
   Agente principal + 3 especialistas · modelo gpt-5

ℹ️  Genero publicaciones de LinkedIn. Temáticas y sus especialistas:
   • Marketing        → 'Haz un post sobre tendencias de email marketing'
   • Programación     → 'Publicación sobre buenas prácticas en Python'
   • Jurídico-Legal   → 'Post sobre las novedades del RGPD'
Mantengo el contexto: puedes pedir '/reiniciar' para empezar de cero.
Comandos:  /ayuda  ·  /reiniciar  ·  /salir

⏳ Agente Principal analizando la solicitud y delegando...
🤝 Procesado por: Agente Jurídico-Legal

────────────────────────────────────────────────────────
📌 RGPD 2024-2025: tres cambios que tu empresa no puede ignorar
────────────────────────────────────────────────────────
La protección de datos no se ha quedado quieta. En 2024-2025, el RGPD convive con nuevas guías, sanciones récord y el aterrizaje del Reglamento de IA de la UE que impacta procesos de datos. ⚖️🔐

Claves recientes que debes tener 